# Assignment 1: LangChain Chains with LCEL
## Modern LangChain Expression Language (LCEL)

**OBJECTIVE:** Build LLM-powered chains using modern LCEL pipe syntax (`prompt | llm | parser`).

**LEARNING GOALS:**
- Understand the LCEL pipe (`|`) operator for composing chains
- Build simple single-step chains
- Wire two chains sequentially by passing outputs as inputs
- Implement multi-step pipelines with named outputs
- Create a practical blog-post generation pipeline

**INSTRUCTIONS:**
1. Complete each function using the inline comment hints as a guide
2. Run each cell after completing the function to test it
3. Each section builds on LCEL concepts — work through them in order


In [ ]:
!pip install langchain_openai langchain_core

In [1]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.7,
    max_tokens=500
)

/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 12/day12/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Simple LCEL Chain

Complete the function below to build a simple chain using the LCEL pipe syntax.

**Concept:** In LCEL, you compose a chain by piping components together with `|`. The output of each component flows into the next.

**Syntax reference:**
```python
prompt = ChatPromptTemplate.from_template("... {variable} ...")
chain  = prompt | llm | StrOutputParser()
result = chain.invoke({"variable": value})
```

**Why it matters:** This is the fundamental building block of all modern LangChain applications.

In [3]:
def create_simple_chain(topic: str) -> str:
    """
    Build a simple LCEL chain that generates a short creative story.

    TODO: Complete this function to create a chain using prompt | llm | StrOutputParser()
    HINT: Use ChatPromptTemplate.from_template() with {topic}, pipe to llm, pipe to StrOutputParser()

    Args:
        topic (str): The topic for the story

    Returns:
        str: The generated story
    """
    # Create a prompt template with a {topic} placeholder
    prompt = ChatPromptTemplate.from_template(
        template="Write a short creative story about {topic}."
    )

    # Pipe the prompt to the llm, then to StrOutputParser()
    chain = prompt | llm | StrOutputParser()

    # Invoke the chain with the topic
    result = chain.invoke({"topic": topic})
    return result


# Test the function after you complete it
story = create_simple_chain("a robot learning to paint")
print(f"Generated Story: {story}")

Generated Story: In a small workshop cluttered with wires, gears, and the soft hum of machinery, lived a robot named Arti. Constructed with the latest in artificial intelligence, Arti was designed to assist in industrial tasks, but a glitch in its programming had sparked an unexpected curiosity. Instead of organizing the workshop or assembling parts, Arti found itself drawn to the far corner where a set of vibrant paints lay forgotten.

One day, while cleaning, Arti stumbled upon a dusty easel and an old canvas. Its sensors whirred with excitement as it examined the colors: reds that glimmered like rubies, blues that whispered of distant oceans, and yellows that radiated warmth like the sun. With a gentle whirr, Arti picked up a brush, its metallic fingers unsure but eager.

At first, the strokes were clumsy, the colors muddled. Arti would dip the brush into a vibrant blue, only to swirl it into a puddle of brown. Frustration bubbled within its circuits, but something deeper—a longing 

## 2. Sequential Chain

Complete the function below to run two chains one after the other, passing the first chain's output as input to the second.

**Concept:** Build each chain separately using `|`. Invoke the first, then pass its string output as a dict value to the second.

**Syntax reference:**
```python
chain1 = prompt1 | llm | StrOutputParser()
chain2 = prompt2 | llm | StrOutputParser()

output1 = chain1.invoke({...})
output2 = chain2.invoke({"key": output1})
```

**Why it matters:** Most real workflows involve multiple LLM calls where the output of one step feeds into the next.

In [4]:
def create_sequential_chain(industry: str) -> str:
    """
    Run two chains in sequence: generate a business idea, then create a marketing slogan for it.

    TODO: Build idea_chain and slogan_chain, invoke idea_chain first, then pass its output to slogan_chain.
    HINT: idea_chain uses {industry}, slogan_chain uses {business_idea}.
          Pass the string output of idea_chain as {"business_idea": ...} to slogan_chain.

    Args:
        industry (str): The industry to generate a business idea for

    Returns:
        str: The marketing slogan for the generated business idea
    """
    # Build the first chain — prompt uses {industry}
    idea_prompt = ChatPromptTemplate.from_template(
        template="Generate a creative business idea in the {industry} industry in less than 20 words."
    )
    idea_chain = idea_prompt | llm | StrOutputParser()

    # Build the second chain — prompt uses {business_idea}
    slogan_prompt = ChatPromptTemplate.from_template(
        template="Create a catchy marketing slogan for this business idea: {business_idea} in less than 10 words."
    )
    slogan_chain = slogan_prompt | llm | StrOutputParser()

    # Run the first chain
    business_idea = idea_chain.invoke({"industry": industry})

    # Pass its output into the second chain
    slogan = slogan_chain.invoke({"business_idea": business_idea})
    return slogan


# Test the function after you complete it
slogan = create_sequential_chain("sustainable technology")
print(f"Final Marketing Slogan: {slogan}")

Final Marketing Slogan: "Grow Smart, Save Water: Urban Farming Reimagined!"


## 3. Multi-Step Pipeline with Named Outputs

Complete the function below to run a 3-step pipeline, passing state between steps manually.

**Concept:** When a later step needs outputs from multiple earlier steps, invoke each chain separately and accumulate results in variables.

**Syntax reference:**
```python
result_a = chain_a.invoke({"x": x})
result_b = chain_b.invoke({"x": x, "a": result_a})
result_c = chain_c.invoke({"x": x, "a": result_a, "b": result_b})
```

**Why it matters:** Complex workflows (analysis, pricing, planning) need multiple named variables flowing between steps.

In [5]:
def run_multi_step_pipeline(product_name: str, target_market: str) -> dict:
    """
    Run a 3-step business analysis pipeline: market analysis -> pricing -> business plan.

    TODO: Create 3 chains, invoke each and pass accumulated results forward.
    HINT: Chain 1 uses {product_name}, {target_market}.
          Chain 2 uses {product_name}, {market_analysis}.
          Chain 3 uses all four: {product_name}, {target_market}, {market_analysis}, {pricing_strategy}.

    Args:
        product_name (str): Name of the product
        target_market (str): Target market description

    Returns:
        dict: Contains 'market_analysis', 'pricing_strategy', and 'business_plan' keys
    """
    # Build and invoke the analysis chain (uses {product_name} and {target_market})
    analysis_prompt = ChatPromptTemplate.from_template(
        template="Perform a market analysis for {product_name} targeting {target_market} in less than 20 words."
    )
    analysis_chain = analysis_prompt | llm | StrOutputParser()
    market_analysis = analysis_chain.invoke({"product_name": product_name, "target_market": target_market})

    # Build and invoke the pricing chain (uses {product_name} and {market_analysis})
    pricing_prompt = ChatPromptTemplate.from_template(
        template="Determine an optimal pricing strategy for {product_name} based on the market analysis: {market_analysis} in less than 20 words."
    )
    pricing_chain = pricing_prompt | llm | StrOutputParser()
    pricing_strategy = pricing_chain.invoke({"product_name": product_name, "market_analysis": market_analysis})

    # Build and invoke the business plan chain (uses all four variables)
    business_plan_prompt = ChatPromptTemplate.from_template(
        template="Create a comprehensive business plan for {product_name} targeting {target_market}, incorporating the market analysis: {market_analysis} and the pricing strategy: {pricing_strategy} in less than 30 words."
    )
    business_plan_chain = business_plan_prompt | llm | StrOutputParser()
    business_plan = business_plan_chain.invoke({
        "product_name": product_name,
        "target_market": target_market,
        "market_analysis": market_analysis,
        "pricing_strategy": pricing_strategy
    })

    return {
        "market_analysis": market_analysis,
        "pricing_strategy": pricing_strategy,
        "business_plan": business_plan
    }


# Test the function after you complete it
result = run_multi_step_pipeline("Smart Fitness Mirror", "health-conscious millennials")
print(f"Market Analysis:\n{result['market_analysis']}")
print(f"\nPricing Strategy:\n{result['pricing_strategy']}")
print(f"\nBusiness Plan:\n{result['business_plan']}")

Market Analysis:
Growing trend in home fitness; millennials seek innovative, tech-driven solutions for convenience and personalized workouts. High market potential.

Pricing Strategy:
Adopt a tiered subscription model with introductory offers, focusing on personalized experiences and community engagement for millennials.

Business Plan:
Launch Smart Fitness Mirror targeting health-conscious millennials with a tiered subscription model, emphasizing personalized workouts, community engagement, and innovative tech solutions in the growing home fitness market.


## 4. Blog Post Creation Pipeline

Complete the function below to build a 3-step blog post pipeline: topics -> outline -> introduction.

**Concept:** Same pattern as Section 3 — build separate chains, invoke sequentially, pass each output forward.

**Why it matters:** Content generation pipelines are one of the most common real-world LLM applications.

In [6]:
def create_blog_pipeline(subject: str) -> dict:
    """
    Build a 3-step blog pipeline: topic generation -> outline -> introduction.

    TODO: Create 3 chains and invoke them sequentially.
    HINT: Chain 1 uses {subject}, Chain 2 uses {topics}, Chain 3 uses {outline}.

    Args:
        subject (str): The subject area for the blog post

    Returns:
        dict: Contains 'topics', 'outline', and 'introduction' keys
    """
    # Build and invoke a chain that generates 3 blog topics from {subject}
    topic_generator = ChatPromptTemplate.from_template(
        template="Generate 3 engaging blog post topics about {subject}."
    ) | llm | StrOutputParser()
    topics = topic_generator.invoke({"subject": subject})

    # Build and invoke a chain that creates an outline from {topics}
    outline_generator = ChatPromptTemplate.from_template(
        template="Create an outline for a blog post with these topics: {topics} in less than 20 words."
    ) | llm | StrOutputParser()
    outline = outline_generator.invoke({"topics": topics})

    # Build and invoke a chain that writes an introduction from {outline}
    intro_writer = ChatPromptTemplate.from_template(
        template="Write an engaging introduction for a blog post with this outline: {outline} in less than 30 words."
    ) | llm | StrOutputParser()
    introduction = intro_writer.invoke({"outline": outline})

    return {"topics": topics, "outline": outline, "introduction": introduction}


# Test the function after you complete it
blog = create_blog_pipeline("artificial intelligence in healthcare")
print(f"Generated Topics:\n{blog['topics']}")
print(f"\nOutline:\n{blog['outline']}")
print(f"\nIntroduction:\n{blog['introduction']}")

Generated Topics:
Sure! Here are three engaging blog post topics about artificial intelligence in healthcare:

1. **"Revolutionizing Patient Care: How AI is Transforming Diagnostics and Treatment Plans"**  
   Explore the latest advancements in AI technologies that are enhancing diagnostic accuracy, personalizing treatment plans, and improving patient outcomes. Highlight real-world case studies where AI has made a significant impact in various medical fields.

2. **"Ethics and AI in Healthcare: Balancing Innovation with Patient Privacy"**  
   Delve into the ethical implications of using AI in healthcare, focusing on data privacy, consent, and the potential biases in AI algorithms. Discuss how healthcare providers can navigate these challenges while leveraging AI for better patient care.

3. **"The Future of Telemedicine: AI's Role in Remote Patient Monitoring and Virtual Consultations"**  
   Investigate how AI is shaping the future of telemedicine, particularly in remote patient moni

## 5. Final Test - Complete Pipeline

Once you've completed all the functions above, run this cell to test the complete pipeline.

In [7]:
print("Testing Complete LCEL Pipeline")
print("=" * 50)

# Step 1: Simple Chain
print("\nStep 1: Simple Chain...")
story = create_simple_chain("a time-traveling chef")
step1_ok = story is not None and len(story) > 10
print(f"   Result: {story[:80]}..." if step1_ok else "   Failed")

# Step 2: Sequential Chain
print("\nStep 2: Sequential Chain...")
slogan = create_sequential_chain("renewable energy")
step2_ok = slogan is not None and len(slogan) > 3
print(f"   Result: {slogan}" if step2_ok else "   Failed")

# Step 3: Multi-Step Pipeline
print("\nStep 3: Multi-Step Pipeline...")
pipeline_result = run_multi_step_pipeline("AI Tutor App", "college students")
step3_ok = (
    pipeline_result is not None
    and "market_analysis" in pipeline_result
    and "pricing_strategy" in pipeline_result
    and "business_plan" in pipeline_result
)
print(f"   Keys returned: {list(pipeline_result.keys())}" if step3_ok else "   Failed")

# Step 4: Blog Pipeline
print("\nStep 4: Blog Pipeline...")
blog = create_blog_pipeline("machine learning for beginners")
step4_ok = (
    blog is not None
    and "topics" in blog
    and "outline" in blog
    and "introduction" in blog
)
print(f"   Keys returned: {list(blog.keys())}" if step4_ok else "   Failed")

# Summary
print("\n" + "=" * 50)
print("Assignment Status:")
print(f"   Simple Chain: {'Pass' if step1_ok else 'Fail'}")
print(f"   Sequential Chain: {'Pass' if step2_ok else 'Fail'}")
print(f"   Multi-Step Pipeline: {'Pass' if step3_ok else 'Fail'}")
print(f"   Blog Pipeline: {'Pass' if step4_ok else 'Fail'}")

if all([step1_ok, step2_ok, step3_ok, step4_ok]):
    print("\nCongratulations! You've successfully completed the assignment!")
    print("You've mastered modern LCEL chain patterns!")
else:
    print("\nPlease complete the TODO functions above to finish the assignment.")

Testing Complete LCEL Pipeline

Step 1: Simple Chain...
   Result: In the bustling heart of Paris, where the aroma of fresh baguettes mingled with ...

Step 2: Sequential Chain...
   Result: "Grow Smart: Harness Sunlight, Optimize Your Harvest!"

Step 3: Multi-Step Pipeline...
   Keys returned: ['market_analysis', 'pricing_strategy', 'business_plan']

Step 4: Blog Pipeline...
   Keys returned: ['topics', 'outline', 'introduction']

Assignment Status:
   Simple Chain: Pass
   Sequential Chain: Pass
   Multi-Step Pipeline: Pass
   Blog Pipeline: Pass

Congratulations! You've successfully completed the assignment!
You've mastered modern LCEL chain patterns!
